# Clasificador de residuos con CNN básica

**Proyecto:** red neuronal convolucional (CNN) propia, construida capa por capa, para clasificar un residuo individual frente a una cámara. Pensado para una futura integración con un robot **Rapiro** ubicado sobre un basurero.

**Clases finales (4):**

| Índice | Clase |
|--------|----------------|
| 0 | plastico |
| 1 | papel_carton |
| 2 | metal |
| 3 | descarte_comun |

La clase **no_identificado** NO se entrena: se aplica luego como regla de confianza (si la probabilidad máxima es menor a un umbral, por ejemplo 0.60, el sistema responde "no_identificado").

**Datasets:** se combinan [Garbage Dataset v2](https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2) (fondos reales y variados) y [RealWaste](https://www.kaggle.com/datasets/joebeachcapital/realwaste) (residuos reales de un centro de tratamiento). Sus clases originales se reagrupan en las 4 clases finales; `clothes` y `shoes` se excluyen porque sus fotos incluyen personas y sesgan al modelo.

**Flujo del notebook:**
1. Preparar el entorno y el código del proyecto.
2. Descargar los datasets.
3. Reagrupar las clases y dividir en train / val / test.
4. Crear la CNN y mostrar su arquitectura.
5. Entrenar con data augmentation y callbacks.
6. Graficar curvas, evaluar y generar matriz de confusión.
7. Probar una imagen individual con la regla de confianza.
8. Guardar el modelo para usarlo luego con la webcam USB local.

## 1. Preparar el entorno y el código del proyecto

El notebook usa los módulos de la carpeta `src/` del proyecto. Hay dos opciones para tener el código disponible en Colab:

- **Opción A (recomendada):** subir la carpeta completa `garbage_cnn_project/` a Google Drive y montar Drive.
- **Opción B:** subir manualmente la carpeta `src/` al entorno de Colab (ícono de carpeta a la izquierda → subir). En ese caso, no hace falta montar Drive, pero los archivos se pierden al cerrar la sesión.

Con la Opción A, el modelo entrenado y los gráficos quedan guardados directamente en tu Drive.

In [ ]:
# --- Opción A: montar Google Drive ---
# Si subiste la carpeta del proyecto a Drive, ejecutá esta celda.
# Si usás la Opción B (subir src/ manualmente), podés saltearla y ajustar
# PROJECT_DIR en la celda siguiente a "/content/garbage_cnn_project".
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# --- Ubicar el proyecto e importar los módulos de src/ ---
import sys
from pathlib import Path

import tensorflow as tf

# Ruta a la carpeta del proyecto. AJUSTAR según dónde esté:
#   Opción A (Drive):  "/content/drive/MyDrive/garbage_cnn_project"
#   Opción B (subida): "/content/garbage_cnn_project"
PROJECT_DIR = Path("/content/drive/MyDrive/garbage_cnn_project")

assert (PROJECT_DIR / "src").is_dir(), (
    f"No se encontró la carpeta src/ en {PROJECT_DIR}. Revisar PROJECT_DIR."
)
sys.path.append(str(PROJECT_DIR / "src"))

from dataset_utils import (
    CLASES_FINALES,
    preparar_dataset,
    guardar_class_indices,
    cargar_class_indices,
    cargar_datasets,
)
from model import IMAGE_SIZE, crear_cnn, crear_data_augmentation
from train import entrenar_modelo, graficar_curvas
from evaluate import (
    evaluar_accuracy,
    generar_matriz_confusion,
    mostrar_predicciones_ejemplo,
    diagnosticar_sobreajuste,
)
from predict_image import predict_image

# Verificar si Colab asignó GPU (Entorno de ejecución -> Cambiar tipo de entorno -> GPU).
# Con GPU el entrenamiento es mucho más rápido, pero también funciona con CPU.
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU") or "No (se usará CPU)")

## 2. Descargar los datasets

Se combinan **dos datasets** complementarios, descargados con `kagglehub` (ya instalado en Colab, **sin necesidad de credenciales**):

1. **[Garbage Dataset v2](https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2)** (~12.300 imágenes, 10 clases): fotos con fondos reales y variados (pisos, mesas, exteriores), lo que hace al modelo mucho más robusto que un dataset de estudio.
2. **[RealWaste](https://www.kaggle.com/datasets/joebeachcapital/realwaste)** (~4.800 imágenes, 9 clases): residuos reales fotografiados en un centro de tratamiento (objetos deformados, sucios, aplastados), tal como se ven los residuos de verdad.

**Nota:** las clases `clothes` y `shoes` se **excluyen a propósito** del entrenamiento. Sus fotos suelen incluir personas (manos, pies), lo que le enseñaba a la red que "piel = descarte_comun" y arruinaba las predicciones con webcam al sostener el residuo con la mano.

**Dataset propio (tercer origen, opcional pero muy recomendado):** si en el proyecto existe la carpeta `dataset_propio/` con fotos capturadas con la webcam del sistema (usando `src/capture_dataset.py` en la PC local), se suma automáticamente al entrenamiento. Estas fotos adaptan la red a la cámara, la iluminación y la escena reales de uso — es el paso que más mejora el rendimiento en el mundo real.

In [ ]:
# --- Descargar los dos datasets de Kaggle ---
import kagglehub

from dataset_utils import encontrar_carpeta_clases

# Cada descarga tiene niveles de carpetas intermedios distintos;
# encontrar_carpeta_clases() localiza la carpeta con las subcarpetas de clases.
DIRS_DATASET_ORIGEN = []
for slug in ("sumn2u/garbage-classification-v2", "joebeachcapital/realwaste"):
    ruta_descarga = Path(kagglehub.dataset_download(slug))
    carpeta_clases = encontrar_carpeta_clases(ruta_descarga)
    DIRS_DATASET_ORIGEN.append(carpeta_clases)
    print(f"\n{slug}")
    print("  Carpeta de clases:", carpeta_clases)
    print("  Clases:", sorted(p.name for p in carpeta_clases.iterdir() if p.is_dir()))

# --- Dataset propio (opcional pero MUY recomendado) ---
# Fotos capturadas con la webcam del sistema usando src/capture_dataset.py.
# Adaptan la red a la cámara, luz y escena reales de uso. Si la carpeta
# dataset_propio/ existe en el proyecto, se suma como tercer origen.
DIR_PROPIO = PROJECT_DIR / "dataset_propio"
if DIR_PROPIO.is_dir():
    DIRS_DATASET_ORIGEN.append(DIR_PROPIO)
    print("\ndataset_propio (fotos con la webcam del sistema)")
    for carpeta in sorted(DIR_PROPIO.iterdir()):
        if carpeta.is_dir():
            print(f"  {carpeta.name:<15} {len(list(carpeta.glob('*.jpg')))} fotos")
else:
    print("\nSin dataset propio (carpeta dataset_propio/ no encontrada). "
          "Se entrena solo con los datasets públicos.")

## 3. Configuración del experimento

Acá se definen todos los parámetros en un solo lugar.

**Cantidad de imágenes:** entre los dos datasets hay ~14.000 imágenes útiles (sin `clothes`/`shoes`), pero después del reagrupamiento `descarte_comun` queda sobrerrepresentada (junta vidrio, orgánico, pilas, trash...). `MAX_IMAGENES_POR_CLASE` controla el tope por clase final:

- `300` → prueba rápida del pipeline (el modelo resultante casi adivina; no sirve para uso real).
- `2500` → **recomendado**: usa casi todos los datos de las clases chicas y recorta `descarte_comun` para mantener el balance.
- `None` → todo; no recomendado por el desbalance hacia `descarte_comun`.

In [ ]:
# --- Parámetros del experimento ---

# Cantidad máxima de imágenes por clase final.
# 300  -> prueba inicial rápida (solo para verificar que el pipeline funciona;
#         el modelo resultante prácticamente adivina)
# 2500 -> entrenamiento real RECOMENDADO: usa casi todas las imágenes de
#         plastico, papel_carton y metal, y recorta descarte_comun (que junta
#         vidrio + orgánico + pilas + trash) para mantener el balance.
# None -> todas las imágenes. NO recomendado: el desbalance hace que la red
#         tienda a responder siempre "descarte_comun".
MAX_IMAGENES_POR_CLASE = 2500

BATCH_SIZE = 32       # imágenes procesadas por paso de entrenamiento
EPOCHS = 20           # épocas máximas (EarlyStopping puede cortar antes)
UMBRAL_CONFIANZA = 0.60  # por debajo de este valor -> "no_identificado"

# Carpetas y archivos de salida.
# El dataset reorganizado va al disco LOCAL de Colab (/content), no a Drive:
# copiar/leer miles de archivos chicos en Drive es lentísimo, y esta carpeta
# se regenera en cada corrida (no hace falta persistirla).
DIR_DATASET_FINAL = Path("/content/dataset_final")
# El modelo y los gráficos sí van a Drive, para que queden guardados.
DIR_MODELS = PROJECT_DIR / "models"
DIR_OUTPUTS = PROJECT_DIR / "outputs"
RUTA_MODELO = DIR_MODELS / "garbage_cnn_model.keras"

# Crear las carpetas de salida si no existen.
DIR_MODELS.mkdir(parents=True, exist_ok=True)
DIR_OUTPUTS.mkdir(parents=True, exist_ok=True)

print("Imagen de entrada:", IMAGE_SIZE)
print("Clases finales:", CLASES_FINALES)
print("Imágenes por clase:", MAX_IMAGENES_POR_CLASE or "todas")

## 4. Reagrupar las clases y dividir en train / val / test

`preparar_dataset()` hace todo el trabajo:

1. Recorre las carpetas de clases de **ambos datasets** y las **reagrupa en las 4 clases finales** según el mapeo definido en `src/dataset_utils.py` (`plastic → plastico`; `paper` y `cardboard → papel_carton`; `metal → metal`; vidrio, orgánico, pilas y trash → `descarte_comun`). Las carpetas `clothes` y `shoes` se ignoran a propósito (aparece un AVISO, es lo esperado).
2. Mezcla las imágenes con una semilla fija (resultado reproducible).
3. Aplica el límite `MAX_IMAGENES_POR_CLASE` si está definido.
4. Divide en **70% train / 15% validación / 15% test**.
5. Muestra cuántas imágenes quedaron por clase y avisa si alguna quedó con pocas.

También se guarda `class_indices.json`, que fija el orden de las clases para que la inferencia futura (imagen individual y webcam) interprete correctamente la salida de la red.

In [ ]:
# --- Combinar ambos datasets en 4 clases y dividir en train / val / test ---
preparar_dataset(
    dir_origen=DIRS_DATASET_ORIGEN,   # lista: [Garbage Dataset v2, RealWaste]
    dir_destino=DIR_DATASET_FINAL,
    max_imagenes_por_clase=MAX_IMAGENES_POR_CLASE,
    proporciones=(0.70, 0.15, 0.15),
    seed=42,
)

# Guardar el orden de clases para la inferencia futura.
guardar_class_indices(DIR_OUTPUTS / "class_indices.json")

In [ ]:
# --- Cargar los datasets como objetos tf.data y ver imágenes de ejemplo ---
import matplotlib.pyplot as plt

train_ds, val_ds, test_ds = cargar_datasets(
    DIR_DATASET_FINAL, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

# Mostrar 9 imágenes de entrenamiento con su etiqueta, para verificar
# que las clases quedaron bien asignadas antes de entrenar.
imagenes, etiquetas = next(iter(train_ds))
plt.figure(figsize=(9, 9))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagenes[i].numpy().astype("uint8"))
    plt.title(CLASES_FINALES[int(etiquetas[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Crear la CNN básica y mostrar su arquitectura

La red está definida capa por capa en `src/model.py`. Conceptualmente:

> El modelo recibe una imagen RGB de 128 × 128 píxeles. Las primeras capas convolucionales extraen patrones visuales simples, como bordes y texturas. Las capas posteriores combinan esos patrones para reconocer formas más complejas asociadas a residuos plásticos, papel/cartón, metales o descarte común. Finalmente, las capas densas integran la información aprendida y la capa softmax entrega una probabilidad para cada clase.

| Capa | Qué hace |
|------|----------|
| `Rescaling` | Normaliza los píxeles de [0, 255] a [0, 1] |
| `Conv2D` (32, 64, 128 filtros) | Aprende filtros que detectan bordes, texturas, formas y patrones |
| `MaxPooling2D` | Reduce el tamaño espacial conservando lo relevante |
| `Flatten` | Convierte los mapas de características en un vector |
| `Dense(128)` | Combina la información aprendida para tomar una decisión |
| `Dropout(0.4)` | Apaga neuronas al azar para evitar sobreajuste |
| `Dense(4) + softmax` | Convierte la salida en probabilidades para cada clase |

`model.summary()` muestra cada capa, la forma de su salida y la cantidad de parámetros entrenables (los "pesos" que la red ajusta durante el aprendizaje).

In [ ]:
# --- Crear la CNN y mostrar el resumen de su arquitectura ---
model = crear_cnn(num_clases=len(CLASES_FINALES))
model.summary()

In [ ]:
# --- (Opcional) Diagrama de la arquitectura con plot_model ---
# Requiere pydot y graphviz. En Colab, graphviz ya está instalado;
# si pydot falta, descomentar la línea siguiente:
# !pip install -q pydot

try:
    tf.keras.utils.plot_model(
        model,
        to_file=str(DIR_OUTPUTS / "model_architecture.png"),
        show_shapes=True,        # muestra la forma de los tensores entre capas
        show_layer_names=True,
        dpi=100,
    )
    from IPython.display import Image, display
    display(Image(str(DIR_OUTPUTS / "model_architecture.png")))
except ImportError as e:
    print("plot_model es opcional y no está disponible:", e)
    print("Instalar con: !pip install pydot   (graphviz ya viene en Colab)")

## 6. Entrenar la red

Durante el entrenamiento, la red ve las imágenes de train (con **data augmentation**: espejado, rotación, zoom y contraste aleatorios) y ajusta sus pesos para reducir el error. Después de cada época se mide el rendimiento sobre validación, **sin** augmentation.

Callbacks utilizados:

- **EarlyStopping:** corta el entrenamiento si la validación deja de mejorar (5 épocas de paciencia) y restaura los mejores pesos.
- **ModelCheckpoint:** guarda automáticamente la mejor versión del modelo en `models/garbage_cnn_model.keras`.
- **ReduceLROnPlateau:** si la validación se estanca, reduce la tasa de aprendizaje a la mitad para afinar el ajuste.

In [ ]:
# --- Entrenamiento (el mejor modelo se guarda automáticamente en RUTA_MODELO) ---
history = entrenar_modelo(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    augmentation=crear_data_augmentation(),
    ruta_modelo=RUTA_MODELO,
    epochs=EPOCHS,
)

## 7. Curvas de entrenamiento y diagnóstico de sobreajuste

- Si **ambas** curvas de accuracy suben juntas → la red está aprendiendo bien.
- Si train sigue subiendo pero validación se estanca o empeora → **sobreajuste** (la red memoriza en lugar de generalizar).

In [ ]:
# --- Graficar accuracy y loss, y diagnosticar sobreajuste ---
graficar_curvas(history, DIR_OUTPUTS / "training_curves.png")
diagnosticar_sobreajuste(history)

## 8. Evaluación del modelo

Primero se **carga el modelo guardado** (la mejor versión según validación, no necesariamente la de la última época) y luego se evalúa:

1. Accuracy en train / validación / test (el de **test** es la medida más honesta: son imágenes que la red nunca vio).
2. **Matriz de confusión:** muestra qué clases se confunden entre sí.
3. **Classification report:** precision, recall y F1-score por clase.
4. Ejemplos visuales de predicciones sobre el set de test.

In [ ]:
# --- Cargar el mejor modelo guardado y evaluar en los tres conjuntos ---
mejor_modelo = tf.keras.models.load_model(RUTA_MODELO)
print(f"Modelo cargado desde: {RUTA_MODELO}\n")

evaluar_accuracy(mejor_modelo, train_ds, val_ds, test_ds)

In [ ]:
# --- Matriz de confusión + classification report (sobre test) ---
generar_matriz_confusion(mejor_modelo, test_ds, DIR_OUTPUTS / "confusion_matrix.png")

In [ ]:
# --- Imágenes de test con su predicción y clase real ---
# Verde = acierto, rojo = error, naranja = no_identificado (confianza < umbral).
mostrar_predicciones_ejemplo(
    mejor_modelo,
    test_ds,
    DIR_OUTPUTS / "sample_predictions.png",
    cantidad=9,
    umbral=UMBRAL_CONFIANZA,
)

## 9. Probar una imagen individual

`predict_image()` aplica la misma lógica que usará el robot:

1. Carga la imagen y la redimensiona a 128 × 128.
2. La pasa por la red (la normalización ocurre dentro del modelo).
3. Obtiene las probabilidades softmax.
4. Si la probabilidad máxima es menor al umbral → devuelve **no_identificado**.

Podés cambiar `ruta_imagen` por cualquier imagen propia subida a Colab o a Drive.

In [ ]:
# --- Predicción de una imagen individual con regla de confianza ---
indice_a_clase = cargar_class_indices(DIR_OUTPUTS / "class_indices.json")

# Por defecto se toma una imagen del set de test; reemplazar por una propia si se desea.
ruta_imagen = next((DIR_DATASET_FINAL / "test" / "plastico").glob("*"))
print("Imagen de prueba:", ruta_imagen.name)

clase, confianza, probabilidades = predict_image(
    ruta_imagen, mejor_modelo, indice_a_clase, umbral=UMBRAL_CONFIANZA
)

print("\nPredicción:")
if clase == "no_identificado":
    print(f"Clase: no_identificado")
    print(f"Confianza máxima: {confianza:.2f} (menor al umbral {UMBRAL_CONFIANZA})")
else:
    print(f"Clase: {clase}")
    print(f"Confianza: {confianza:.2f}")

print("\nProbabilidades por clase:")
for indice, nombre in sorted(indice_a_clase.items()):
    print(f"  {nombre:<16} {probabilidades[indice]:.4f}")

plt.imshow(plt.imread(ruta_imagen))
plt.title(f"Predicción: {clase} ({confianza:.2f})")
plt.axis("off")
plt.show()

## 10. Descargar el modelo y próximos pasos

Si trabajaste con la **Opción A (Drive)**, el modelo y los resultados ya quedaron guardados en tu Drive:

- `models/garbage_cnn_model.keras` → modelo entrenado
- `outputs/class_indices.json` → orden de clases (necesario para inferir)
- `outputs/training_curves.png`, `outputs/confusion_matrix.png`, `outputs/sample_predictions.png`

Si usaste la Opción B, ejecutá la celda siguiente para descargar el modelo a tu PC.

### Uso con webcam USB (en tu PC, no en Colab)

Colab corre en un servidor remoto y **no puede acceder a la webcam USB local**. Para la inferencia en tiempo real, copiá el proyecto (con el modelo entrenado y `class_indices.json`) a tu PC o Raspberry Pi y ejecutá:

```bash
pip install -r requirements.txt
python src/webcam_inference.py
```

Ver el README para más detalles y para la proyección de integración con el robot Rapiro.

In [ ]:
# --- (Solo Opción B) Descargar el modelo y class_indices.json a la PC ---
from google.colab import files

files.download(str(RUTA_MODELO))
files.download(str(DIR_OUTPUTS / "class_indices.json"))